In [1]:
from bcbench.config import get_config
from bcbench.dataset import DatasetEntry, load_dataset_entries

_config = get_config()

bcbench_dataset: list[DatasetEntry] = load_dataset_entries(_config.paths.dataset_path)

print(f"Total number of instances: {len(bcbench_dataset)}")

Total number of instances: 72


In [2]:
from collections import Counter

import pandas as pd

project_counts: Counter[str] = Counter([entry.extract_project_name() for entry in bcbench_dataset])
project_df = pd.DataFrame(project_counts.items(), columns=["Project", "Count"]).sort_values("Count", ascending=False)
project_df["Percentage"] = (project_df["Count"] / project_df["Count"].sum() * 100).round(1)

print("Projects Distribution:")
print(project_df.to_string(index=False))

Projects Distribution:
                   Project  Count  Percentage
                   BaseApp     57        79.2
                   Shopify      7         9.7
            Sustainability      2         2.8
              ExcelReports      2         2.8
EssentialBusinessHeadlines      1         1.4
   EnforcedDigitalVouchers      1         1.4
      Subscription Billing      1         1.4
       SubscriptionBilling      1         1.4


In [3]:
area_counts: Counter[str] = Counter([entry.metadata.area if entry.metadata.area else "Unknown" for entry in bcbench_dataset])

area_df = pd.DataFrame(area_counts.items(), columns=["Area", "Count"]).sort_values("Count", ascending=False)
area_df["Percentage"] = (area_df["Count"] / area_df["Count"].sum() * 100).round(1)

print("\nArea Distribution:")
print(area_df.to_string(index=False))


Area Distribution:
                Area  Count  Percentage
           inventory     13        18.1
             finance     11        15.3
               sales     10        13.9
             shopify      7         9.7
           warehouse      5         6.9
             project      5         6.9
       manufacturing      4         5.6
             service      3         4.2
      sustainability      2         2.8
           reporting      2         2.8
                 crm      2         2.8
subscription billing      2         2.8
            eservice      1         1.4
       visualization      1         1.4
             pricing      1         1.4
        intercompany      1         1.4
           utilities      1         1.4
            assembly      1         1.4


In [4]:
from unidiff import PatchSet

gold_patch_sets: list[PatchSet] = [PatchSet(entry.patch) for entry in bcbench_dataset]

files_changed_counts: Counter[int] = Counter([len(patch_set) for patch_set in gold_patch_sets])

files_changed_df = pd.DataFrame(files_changed_counts.items(), columns=["Number of Files Changed", "Count"]).sort_values("Number of Files Changed")
files_changed_df["Percentage"] = (files_changed_df["Count"] / files_changed_df["Count"].sum() * 100).round(1)

print(files_changed_df.to_string(index=False))

 Number of Files Changed  Count  Percentage
                       1     61        84.7
                       2      8        11.1
                       3      1         1.4
                       4      1         1.4
                       6      1         1.4


In [5]:
lines_of_code_changed: list[int] = [patch_set.added + patch_set.removed for patch_set in gold_patch_sets]

bins = [0, 5, 10, 20, 50, 100, float("inf")]
labels = ["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
bucketed = pd.cut(lines_of_code_changed, bins=bins, labels=labels)

lines_of_code_changed_df = pd.DataFrame({"LoC": bucketed}).value_counts().reset_index(name="Count")
lines_of_code_changed_df = lines_of_code_changed_df.sort_values("LoC")
lines_of_code_changed_df["Percentage"] = (lines_of_code_changed_df["Count"] / lines_of_code_changed_df["Count"].sum() * 100).round(1)

print(lines_of_code_changed_df.to_string(index=False))

   LoC  Count  Percentage
   1-5     30        41.7
  6-10      8        11.1
 11-20     13        18.1
 21-50     13        18.1
51-100      5         6.9
  100+      3         4.2


In [6]:
from statistics import mean

files_changed: list[int] = [len(patch_set) for patch_set in gold_patch_sets]

print("Solution requires:")
print(f"~{mean(lines_of_code_changed):.1f} LoC on average (stdev: {pd.Series(lines_of_code_changed).std():.1f}, median: {pd.Series(lines_of_code_changed).median():.1f})")
print(f"~{mean(files_changed):.1f} files on average (stdev: {pd.Series(files_changed).std():.1f}, median: {pd.Series(files_changed).median():.1f})")

Solution requires:
~19.2 LoC on average (stdev: 26.5, median: 10.0)
~1.2 files on average (stdev: 0.8, median: 1.0)


In [7]:
image_counts: list[int] = [entry.metadata.image_count or 0 for entry in bcbench_dataset]

bins = [-1, 0, 5, 10, float("inf")]
labels = ["0", "1-5", "6-10", "10+"]
bucketed = pd.cut(image_counts, bins=bins, labels=labels)

image_count_df = pd.DataFrame({"Image Count": bucketed}).value_counts().reset_index(name="Count")
image_count_df = image_count_df.sort_values("Image Count")
image_count_df["Percentage"] = (image_count_df["Count"] / image_count_df["Count"].sum() * 100).round(1)

print(image_count_df.to_string(index=False))

Image Count  Count  Percentage
          0     29        40.3
        1-5     18        25.0
       6-10     17        23.6
        10+      8        11.1
